In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

ImportError: DLL load failed while importing _pywrap_profiler_plugin: An Application Control policy has blocked this file.

In [3]:
df=pd.read_csv("./Data/data_MBTI.csv")
df.head()

,Age,Gender,Education,Introversion Score,Sensing Score,Thinking Score,Judging Score,Interest,Personality
0,19,Male,0,9.47080,7.141434,6.03696,4.360278,Unknown,ENFP
1,27,Female,0,5.85392,6.160195,0.80552,4.221421,Sports,ESFP
2,21,Female,0,7.08615,3.388433,2.66188,5.127320,Unknown,ENFP
3,28,Male,0,2.01892,4.823624,7.30625,5.986550,Others,INTP
4,36,Female,1,9.91703,4.755080,5.31469,4.677213,Technology,ENFP


In [ ]:
# Separate Features and Target
X = df.drop('Personality', axis=1)
y = df['Personality']

In [ ]:
# Define column types
categorical_features = ['Gender', 'Interest']
numerical_features = ['Age', 'Education', 'Introversion Score', 'Sensing Score', 'Thinking Score', 'Judging Score']

In [ ]:
# Encode the Target (Personality Types)
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)
num_classes = len(le_y.classes_)
print(num_classes)

In [11]:
# Preprocessing Pipeline for Features
# This scales numbers and converts text (Gender/Interest) into numbers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

X_preprocessed = preprocessor.fit_transform(X)

In [12]:
# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed, y_encoded, test_size=0.2, random_state=42
)

In [19]:
# --- STEP 3: BUILD THE TENSORFLOW MODEL ---
mmodel = tf.keras.models.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),  # ✅ Add this as first layer
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [20]:
# --- STEP 4: TRAIN THE MODEL ---
print("\nStarting Training...")
history = model.fit(
    X_train, y_train, 
    epochs=15, 
    batch_size=32, 
    validation_split=0.2, 
    verbose=1
)

# --- STEP 5: EVALUATE ---
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")


Starting Training...
Epoch 1/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.8919 - loss: 0.2168 - val_accuracy: 0.8853 - val_loss: 0.2168
Epoch 2/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.8912 - loss: 0.2175 - val_accuracy: 0.8901 - val_loss: 0.2115
Epoch 3/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.8924 - loss: 0.2155 - val_accuracy: 0.8940 - val_loss: 0.2064
Epoch 4/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.8919 - loss: 0.2153 - val_accuracy: 0.8917 - val_loss: 0.2099
Epoch 5/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.8925 - loss: 0.2147 - val_accuracy: 0.8945 - val_loss: 0.2078
Epoch 6/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.8920 - loss: 0.2146 - val_accuracy: 0.8946 - val_loss: 0.2041
Epoch 7/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.8910 - loss: 0.2145 - val_accuracy: 0.8913 - val_loss: 0.2126
Epoch 8/15
2562/2562 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accu

In [22]:
# --- STEP 6: MAKE A PREDICTION ---
def predict_personality(age, gender, education, intro, sensing, thinking, judging, interest):
    # Create a small dataframe for the input
    input_data = pd.DataFrame([[age, gender, education, intro, sensing, thinking, judging, interest]], 
    columns=['Age', 'Gender', 'Education', 'Introversion Score', 
                                       'Sensing Score', 'Thinking Score', 'Judging Score', 'Interest'])
    
    # Preprocess the input
    input_scaled = preprocessor.transform(input_data)
    
    # Predict
    prediction = model.predict(input_scaled)
    predicted_class = np.argmax(prediction)
    
    return le_y.inverse_transform([predicted_class])[0]

In [23]:
# Example Test
print("\nTesting Prediction Logic:")
test_result = predict_personality(25, 'Female', 1, 8.5, 4.2, 5.0, 3.1, 'Technology')
print(f"Predicted Personality: {test_result}")


Testing Prediction Logic:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
Predicted Personality: ENFP
